In [90]:
import pandas as pd
import numpy as np
import duckdb

# Data Preparation

In [91]:
orders = pd.read_csv("data/olist_orders_dataset.csv")
customers = pd.read_csv("data/olist_customers_dataset.csv")
payments = pd.read_csv("data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")
items = pd.read_csv("data/olist_order_items_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")
translation = pd.read_csv("data/product_category_name_translation.csv")
sellers = pd.read_csv("data/olist_sellers_dataset.csv")
geo = pd.read_csv("data/olist_geolocation_dataset.csv")

## Missing values

In [92]:
data = {
    "Orders": orders,
    "Customers": customers,
    "Payments": payments,
    "Reviews": reviews,
    "Items": items,
    "Products": products,
    "Translation": translation,
    "Sellers": sellers,
    "Geo": geo
}

for name, df in data.items():
    print(f"=== {name} ===")
    
    null_counts = df.isnull().sum()
    missing = null_counts[null_counts > 0]
    
    if missing.empty:
        print("No missing values.\n")
    else:
        print(missing)
        print("\n")

=== Orders ===
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


=== Customers ===
No missing values.

=== Payments ===
No missing values.

=== Reviews ===
review_comment_title      87656
review_comment_message    58247
dtype: int64


=== Items ===
No missing values.

=== Products ===
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


=== Translation ===
No missing values.

=== Sellers ===
No missing values.

=== Geo ===
No missing values.



# Data dup

In [93]:
for name, df in data.items():
    duplicate_count = df.duplicated().sum()
    
    print(f"=== {name} ===")
    if duplicate_count == 0:
        print("No duplicate rows.\n")
    else:
        print(f"Found {duplicate_count} duplicate row(s).\n")

=== Orders ===
No duplicate rows.

=== Customers ===
No duplicate rows.

=== Payments ===
No duplicate rows.

=== Reviews ===
No duplicate rows.

=== Items ===
No duplicate rows.

=== Products ===
No duplicate rows.

=== Translation ===
No duplicate rows.

=== Sellers ===
No duplicate rows.

=== Geo ===
Found 261831 duplicate row(s).



In [94]:
geo.duplicated().sum()

np.int64(261831)

In [95]:
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

In [96]:
orders_delivered.isnull().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

In [ ]:
orders_delivered.dropna(subset=[
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date'
], inplace=True)

#Verify 
print(orders_delivered.isnull().sum())

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64


In [98]:
#review
reviews['review_comment_title'] = reviews['review_comment_title'].fillna("No Title")
reviews['review_comment_message'] = reviews['review_comment_message'].fillna("No Message")

#product 
products['product_category_name'] = products['product_category_name'].fillna("Unknown")
zero_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty']
products[zero_cols] = products[zero_cols].fillna(0)
median_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in median_cols:
    products[col] = products[col].fillna(products[col].median())

print("Reviews missing values:", reviews.isnull().sum().sum())
print("Products missing values:", products.isnull().sum().sum())

Reviews missing values: 0
Products missing values: 0


In [99]:
geo_unique = geo.drop_duplicates(subset=['geolocation_zip_code_prefix'])

In [100]:
geo_unique.duplicated().sum()

np.int64(0)

# JOIN

In [ ]:
query = """
    SELECT 
        o.order_id, 
        o.customer_id, 
        o.order_status, 
        o.order_purchase_timestamp, 
        o.order_approved_at, 
        o.order_delivered_carrier_date, 
        o.order_delivered_customer_date, 
        o.order_estimated_delivery_date,

        c.customer_unique_id, 
        c.customer_zip_code_prefix, 
        c.customer_city, 
        c.customer_state,

        p.payment_sequential, 
        p.payment_type, 
        p.payment_installments, 
        p.payment_value,

        r.review_id, 
        r.review_score, 
        r.review_comment_title, 
        r.review_comment_message, 
        r.review_creation_date, 
        r.review_answer_timestamp,

        i.order_item_id, 
        i.product_id, 
        i.seller_id, 
        i.shipping_limit_date, 
        i.price, 
        i.freight_value,

        pr.product_category_name, 
        pr.product_name_lenght, 
        pr.product_description_lenght, 
        pr.product_photos_qty, 
        pr.product_weight_g, 
        pr.product_length_cm, 
        pr.product_height_cm, 
        pr.product_width_cm,

        tr.product_category_name_english,

        s.seller_zip_code_prefix, 
        s.seller_city, 
        s.seller_state,

        gc.geolocation_lat AS customer_lat, 
        gc.geolocation_lng AS customer_lng,
        gs.geolocation_lat AS seller_lat, 
        gs.geolocation_lng AS seller_lng

    FROM orders_delivered o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    LEFT JOIN payments p ON o.order_id = p.order_id
    LEFT JOIN reviews r ON o.order_id = r.order_id
    LEFT JOIN items i ON o.order_id = i.order_id
    LEFT JOIN products pr ON i.product_id = pr.product_id
    LEFT JOIN translation tr ON pr.product_category_name = tr.product_category_name
    LEFT JOIN sellers s ON i.seller_id = s.seller_id
    LEFT JOIN geo_unique gc ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
    LEFT JOIN geo_unique gs ON s.seller_zip_code_prefix = gs.geolocation_zip_code_prefix
"""

df = duckdb.query(query).df()

In [105]:
print("=== Merged Dataset ===")

null_counts = df.isnull().sum()
missing = null_counts[null_counts > 0]

print(missing)

=== Merged Dataset ===
review_id                  861
review_creation_date       861
review_answer_timestamp    861
customer_lat               303
customer_lng               303
seller_lat                 261
seller_lng                 261
dtype: int64


In [ ]:
df.dropna(subset=['payment_value'], inplace=True)

df['product_category_name_english'] = df['product_category_name_english'].fillna('Unknown')

df['review_score'] = df['review_score'].fillna(0)
df['review_comment_title'] = df['review_comment_title'].fillna('No Title')
df['review_comment_message'] = df['review_comment_message'].fillna('No Message')

In [106]:
print(missing)

review_id                  861
review_creation_date       861
review_answer_timestamp    861
customer_lat               303
customer_lng               303
seller_lat                 261
seller_lng                 261
dtype: int64


In [107]:
df.to_csv("data/merged_dataset.csv", index=False)